This was code to extract weather data for the two villages.

We ultimately decided against using it within the manuscript. 

however I will keep it here in case people are interested.

In [ ]:
import ee

# Initialize the Earth Engine library
ee.Initialize()

# Define two points
gniby = ee.Geometry.Point([-15.682083368564234, 14.460248196957838])
gossas = ee.Geometry.Point([-16.079953061378422, 14.533427928227212])

# Create a 5 km buffer around each point
gniby_buffer1 = gniby.buffer(5000)
gossas_buffer2 = gossas.buffer(5000)

# Load the ERA5 Daily dataset
era5 = ee.ImageCollection("ECMWF/ERA5/DAILY")

# Define date range
start_date = '2019-01-01'
end_date = '2019-12-31'
era5_filtered = era5.filterDate(start_date, end_date)

# Select climate variables
variables = ['total_precipitation', 'mean_2m_air_temperature', 'minimum_2m_air_temperature', 'maximum_2m_air_temperature']
era5_selected = era5_filtered.select(variables)

# Function to extract mean values for each buffer
def extract_data(image, buffer, label):
    stats = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=buffer,
        scale=1000,
        bestEffort=True
    )
    return image.set(stats).set({'date': image.date().format(), 'location': label})

# Apply extraction separately for each coordinate
era5_gniby = era5_selected.map(lambda image: extract_data(image, gniby_buffer1, 'Point1'))
era5_gossas = era5_selected.map(lambda image: extract_data(image, gossas_buffer2, 'Point2'))

# Merge the results
era5_processed = era5_gniby.merge(era5_gossas)

# Convert to Feature Collection
results = era5_processed.map(lambda image: ee.Feature(None, image.toDictionary()))

# Print the extracted data
print(results.getInfo())

# (Optional) Export results as CSV
export_task = ee.batch.Export.table.toDrive(
    collection=results,
    description='ERA5_Weather_Data_Per_Point',
    fileFormat='CSV'
)
export_task.start()


Exception: Problem requesting tokens. Please try again.  HTTP Error 400: Bad Request b'{\n  "error": "invalid_request",\n  "error_description": "Missing required parameter: code"\n}'

In [ ]:
# Authenticate and initialize GEE
ee.Initialize()

# Define the two coordinates (latitude, longitude)
coords = [
    ee.Geometry.Point([lon1, lat1]),  # Replace with actual values
    ee.Geometry.Point([lon2, lat2])   # Replace with actual values
]

# Create a 5km buffer around each point
buffers = [point.buffer(5000) for point in coords]  # 5000 meters = 5 km

# Merge the buffers into a single feature collection
region = ee.Geometry.MultiPolygon([buf.geometry().coordinates() for buf in buffers])

# Load the ERA5 Daily dataset
era5 = ee.ImageCollection("ECMWF/ERA5/DAILY")

# Define the time range
start_date = '2023-01-01'  # Change as needed
end_date = '2023-12-31'    # Change as needed
era5_filtered = era5.filterDate(start_date, end_date)

# Select variables: precipitation, temperature, and humidity
variables = ['total_precipitation', 'mean_2m_air_temperature', 'mean_2m_relative_humidity']
era5_selected = era5_filtered.select(variables)

# Reduce the dataset over the region using mean reduction
def extract_data(image):
    stats = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=1000,  # Adjust scale as needed
        bestEffort=True
    )
    return image.set(stats).set('date', image.date().format())

# Apply extraction function
era5_processed = era5_selected.map(extract_data)

# Convert results to a Pandas DataFrame
import pandas as pd

# Get the results as a list
data_list = era5_processed.aggregate_array('date').getInfo()
values = era5_processed.aggregate_array(variables).getInfo()

# Convert to DataFrame
df = pd.DataFrame(values, index=data_list)
df.index = pd.to_datetime(df.index)
df.columns = variables
print(df)
